# 04 — ASR (Automatic Speech Recognition)

**Purpose:** Transcribe source-language speech to text, aligned to speaker-turn timestamps from diarization.

## Model selection rationale

| Model | Architecture | Why included |
|---|---|---|
| **IndicWhisper (AI4Bharat)** | Whisper fine-tune on ~10 000 h Indic audio | Best coverage of Tamil/Hindi phonology; handles code-switching, Dravidian prosody |
| **Whisper Large-V3** | Transformer seq2seq, 1.5B params | OpenAI baseline; strong multilingual; reference for pseudo-WER |
| **WhisperX** | Batched Whisper + forced alignment | 70x faster than real-time on GPU; provides accurate word-level timestamps |
| **NVIDIA Canary-1B** | FastConformer encoder + mT5 decoder | Strong on South Asian languages; small model size relative to quality |
| **Sarvam-1 (Sarvam AI)** | Whisper fine-tune on Indic languages | Specifically trained on Dravidian; strong Tamil, Telugu, Kannada support |
| **Gemini ASR** | Native audio understanding LLM | Processes full audio holistically; excellent on Indic; used as pseudo-GT reference |

## Metric rationale
- **WER (Word Error Rate)** = (insertions + deletions + substitutions) / reference_words. Lower is better. Requires a reference transcript.
- **CER (Character Error Rate)** = same metric at character level. More meaningful for agglutinative Dravidian languages where word boundaries are fuzzy.
- **Pseudo-WER**: We use Gemini's transcript as a "pseudo ground truth" since we lack a human reference. Any model with lower pseudo-WER means it agrees more with Gemini — not perfect but useful for ranking.
- **ASR confidence scores** (`avg_logprob`, `no_speech_prob`): saved per segment. Used in downstream ensembling and to flag low-confidence segments for manual review.

## Ensemble strategy
For the canonical transcript saved to `transcription.json`, a simple approach is: for each segment, use the model with the **highest avg_logprob**. A more robust approach (ROVER) requires additional tooling.

**API keys needed:** `GEMINI_API_KEY`. See `API_KEYS.md`.

**Input:** `stems/vocals.wav` + `diarization/diarization.json`  
**Output:** `transcription/transcription.json`


In [ ]:
!pip install -q openai-whisper transformers accelerate jiwer tqdm pandas google-generativeai
print('Ready.')


In [ ]:
import sys, os, json, time
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
from getpass import getpass
import torch
import numpy as np
import librosa
import pandas as pd
from tqdm.notebook import tqdm

sys.path.insert(0, os.path.abspath('..'))
from config import (
    STEMS_DIR, DIARIZATION_DIR, TRANSCRIPTION_DIR,
    ASR_SAMPLE_RATE, SOURCE_LANGUAGE, LANGUAGE_DISPLAY,
    WHISPER_LANGUAGE_CODE, GEMINI_PRO_MODEL, get_torch_device,
)

VOCALS_WAV = os.path.join(STEMS_DIR, 'vocals.wav')
DIAR_JSON  = os.path.join(DIARIZATION_DIR, 'diarization.json')
OUT_JSON   = os.path.join(TRANSCRIPTION_DIR, 'transcription.json')

with open(DIAR_JSON) as f:
    diar_segs = json.load(f)
print(f'Diarization: {len(diar_segs)} segments')

y_asr, _ = librosa.load(VOCALS_WAV, sr=ASR_SAMPLE_RATE, mono=True)
DURATION  = len(y_asr) / ASR_SAMPLE_RATE
print(f'Audio (16 kHz for ASR): {DURATION:.1f}s')

# Device detection: CUDA > MPS (Apple Silicon) > CPU
DEVICE, DTYPE = get_torch_device()
COMPUTE_TYPE  = 'float16' if DEVICE in ('cuda', 'mps') else 'int8'
print(f'Device: {DEVICE}  dtype: {DTYPE}  whisperx_compute: {COMPUTE_TYPE}')

LANG = WHISPER_LANGUAGE_CODE.get(SOURCE_LANGUAGE, SOURCE_LANGUAGE)
all_results = {}

def slice_16k(start, end):
    s = int(start * ASR_SAMPLE_RATE)
    e = int(end   * ASR_SAMPLE_RATE)
    return y_asr[s:e]

def best_overlap_speaker(seg_start, seg_end, diar_segs):
    best_spk, best_ov = 'UNKNOWN', 0.0
    for d in diar_segs:
        ov = max(0, min(seg_end, d['end']) - max(seg_start, d['start']))
        if ov > best_ov:
            best_ov = ov
            best_spk = d['speaker']
    return best_spk

## OSS Models


In [ ]:
# ── IndicWhisper (AI4Bharat) ──────────────────────────────────────────────────
try:
    from transformers import WhisperProcessor, WhisperForConditionalGeneration

    MODEL_ID  = 'ai4bharat/indicwhisper'
    print(f'Loading IndicWhisper on {DEVICE}...')
    t0 = time.time()
    proc_iw  = WhisperProcessor.from_pretrained(MODEL_ID)
    model_iw = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
    print(f'Loaded in {time.time()-t0:.1f}s')

    forced_ids = proc_iw.get_decoder_prompt_ids(language=SOURCE_LANGUAGE, task='transcribe')

    segs_iw = []
    t0 = time.time()
    for seg in tqdm(diar_segs, desc='IndicWhisper'):
        chunk = slice_16k(seg['start'], seg['end'])
        if len(chunk) < 160:
            segs_iw.append({'speaker': seg['speaker'], 'start': seg['start'],
                            'end': seg['end'], 'text': '', 'avg_logprob': None, 'no_speech_prob': None})
            continue
        inputs = proc_iw(chunk, sampling_rate=ASR_SAMPLE_RATE, return_tensors='pt').input_features.to(DEVICE)
        if DTYPE != torch.float32:
            inputs = inputs.to(DTYPE)
        with torch.no_grad():
            out = model_iw.generate(inputs, forced_decoder_ids=forced_ids)
        text = proc_iw.batch_decode(out, skip_special_tokens=True)[0].strip()
        segs_iw.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                        'text': text, 'avg_logprob': None, 'no_speech_prob': None})

    elapsed = time.time() - t0
    all_results['IndicWhisper'] = segs_iw
    print(f'IndicWhisper: {len(segs_iw)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except Exception as e:
    print(f'IndicWhisper FAILED: {e}')

In [ ]:
# ── Whisper Large-V3 (OpenAI baseline) ───────────────────────────────────────
try:
    import whisper

    print(f'Loading Whisper large-v3 on {DEVICE}...')
    t0 = time.time()
    model_wv3 = whisper.load_model('large-v3', device=DEVICE)
    print(f'Loaded in {time.time()-t0:.1f}s')

    segs_wv3 = []
    t0 = time.time()
    for seg in tqdm(diar_segs, desc='Whisper-LargeV3'):
        chunk = slice_16k(seg['start'], seg['end']).astype(np.float32)
        if len(chunk) < 160:
            segs_wv3.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                             'text': '', 'avg_logprob': None, 'no_speech_prob': 1.0})
            continue
        res = model_wv3.transcribe(chunk, language=SOURCE_LANGUAGE, task='transcribe',
                                   fp16=(DEVICE in ('cuda', 'mps')))
        sub    = res.get('segments', [])
        avg_lp = float(np.mean([s.get('avg_logprob', 0) for s in sub])) if sub else None
        nsp    = float(np.mean([s.get('no_speech_prob', 0) for s in sub])) if sub else None
        segs_wv3.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                         'text': res['text'].strip(), 'avg_logprob': avg_lp, 'no_speech_prob': nsp})

    elapsed = time.time() - t0
    all_results['Whisper-LargeV3'] = segs_wv3
    print(f'Whisper LargeV3: {len(segs_wv3)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except Exception as e:
    print(f'Whisper LargeV3 FAILED: {e}')

In [ ]:
# ── WhisperX — batched, word-level timestamps ─────────────────────────────────
try:
    import whisperx

    lang = WHISPER_LANGUAGE_CODE.get(SOURCE_LANGUAGE, SOURCE_LANGUAGE)
    print(f'WhisperX large-v3: device={DEVICE}, compute={COMPUTE_TYPE}, lang={lang}')
    t0 = time.time()
    model_wx  = whisperx.load_model('large-v3', DEVICE, compute_type=COMPUTE_TYPE)
    audio_wx  = whisperx.load_audio(VOCALS_WAV)
    result    = model_wx.transcribe(audio_wx, batch_size=16, language=lang)
    elapsed   = time.time() - t0

    segs_wx = []
    for s in result['segments']:
        sp = best_overlap_speaker(s['start'], s['end'], diar_segs)
        segs_wx.append({'speaker': sp, 'start': s['start'], 'end': s['end'],
                        'text': s['text'].strip(), 'avg_logprob': s.get('avg_logprob'), 'no_speech_prob': s.get('no_speech_prob')})
    all_results['WhisperX'] = segs_wx
    print(f'WhisperX: {len(segs_wx)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except Exception as e:
    print(f'WhisperX FAILED: {e}')

In [ ]:
# ── Sarvam-1 (Sarvam AI) — Indic-specialized ─────────────────────────────────
try:
    from transformers import pipeline as hf_pipeline

    print(f'Loading Sarvam-1 on {DEVICE}...')
    t0 = time.time()
    pipe_sarvam = hf_pipeline(
        'automatic-speech-recognition',
        model='sarvamai/sarvam-1',
        device=DEVICE,
        torch_dtype=DTYPE,
    )
    print(f'Loaded in {time.time()-t0:.1f}s')

    segs_sarvam = []
    t0 = time.time()
    for seg in tqdm(diar_segs, desc='Sarvam-1'):
        chunk = slice_16k(seg['start'], seg['end'])
        if len(chunk) < 160:
            segs_sarvam.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                                'text': '', 'avg_logprob': None, 'no_speech_prob': None})
            continue
        res = pipe_sarvam({'array': chunk.astype(np.float32), 'sampling_rate': ASR_SAMPLE_RATE})
        segs_sarvam.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                            'text': res['text'].strip(), 'avg_logprob': None, 'no_speech_prob': None})
    elapsed = time.time() - t0
    all_results['Sarvam-1'] = segs_sarvam
    print(f'Sarvam-1: {len(segs_sarvam)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except Exception as e:
    print(f'Sarvam-1 FAILED (model may not be public yet): {e}')

In [ ]:
# ── NVIDIA Canary-1B (FastConformer + mT5 decoder) ────────────────────────────
# 1B-parameter model from NVIDIA NeMo. FastConformer encoder is significantly
# more efficient than Whisper's attention encoder. Good on South Asian languages.
try:
    import nemo.collections.asr as nemo_asr
    import torch

    t0 = time.time()
    model_canary = nemo_asr.models.EncDecMultiTaskModel.from_pretrained('nvidia/canary-1b')
    model_canary.eval()
    print(f'Canary-1B loaded in {time.time()-t0:.1f}s')

    decode_cfg = model_canary.cfg.decoding
    decode_cfg.beam.beam_size = 1
    model_canary.change_decoding_strategy(decode_cfg)

    segs_canary = []
    t0 = time.time()
    for seg in tqdm(diar_segs, desc='Canary-1B'):
        chunk = slice_16k(seg['start'], seg['end']).astype(np.float32)
        if len(chunk) < 160:
            segs_canary.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                                'text': '', 'avg_logprob': None, 'no_speech_prob': None})
            continue
        # Canary expects a list of audio arrays + source_lang + target_lang
        preds = model_canary.transcribe(
            [chunk],
            batch_size=1,
            task='asr',
            source_lang=SOURCE_LANGUAGE,
            target_lang=SOURCE_LANGUAGE,
        )
        text = preds[0] if preds else ''
        segs_canary.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                            'text': text.strip(), 'avg_logprob': None, 'no_speech_prob': None})
    elapsed = time.time() - t0
    all_results['Canary-1B'] = segs_canary
    print(f'Canary-1B: {len(segs_canary)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except ImportError:
    print('NeMo not installed. Run: pip install nemo_toolkit[asr]  (large install)')
except Exception as e:
    print(f'Canary-1B FAILED: {e}')

In [ ]:
# ── Collabora Whisper-Hindi v2 (Hindi runs only) ──────────────────────────────
# Fine-tune of Whisper on a large Hindi dataset. Only makes sense on Hindi source.
# When SOURCE_LANGUAGE == 'ta' (Tamil), this cell skips automatically.
if SOURCE_LANGUAGE != 'hi':
    print(f'Collabora Whisper-Hindi: skipped (source is {SOURCE_LANGUAGE}, not hi).')
    print('This model runs automatically when you switch config to SOURCE_LANGUAGE = "hi".')
else:
    try:
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        import torch

        device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
        MODEL_ID = 'collabora/whisper-large-v2-hindi'
        print(f'Loading {MODEL_ID} on {device}...')
        t0 = time.time()
        proc_col  = WhisperProcessor.from_pretrained(MODEL_ID)
        model_col = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
        print(f'Loaded in {time.time()-t0:.1f}s')

        forced_ids = proc_col.get_decoder_prompt_ids(language='hi', task='transcribe')
        segs_col   = []
        t0 = time.time()
        for seg in tqdm(diar_segs, desc='Collabora Whisper-Hindi'):
            chunk = slice_16k(seg['start'], seg['end'])
            if len(chunk) < 160:
                segs_col.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                                 'text': '', 'avg_logprob': None, 'no_speech_prob': None})
                continue
            inputs = proc_col(chunk, sampling_rate=ASR_SAMPLE_RATE, return_tensors='pt').input_features.to(device)
            with torch.no_grad():
                out = model_col.generate(inputs, forced_decoder_ids=forced_ids)
            text = proc_col.batch_decode(out, skip_special_tokens=True)[0].strip()
            segs_col.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                             'text': text, 'avg_logprob': None, 'no_speech_prob': None})
        elapsed = time.time() - t0
        all_results['Collabora-Whisper-Hindi'] = segs_col
        print(f'Collabora Whisper-Hindi: {len(segs_col)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    except Exception as e:
        print(f'Collabora Whisper-Hindi FAILED: {e}')

## OSS — MOSS-Audio (OpenMOSS)

Audio **understanding** foundation model — takes audio in, produces text output. Not a TTS or stem separator. Relevant here for ASR: claims best-in-class CER with native timestamp awareness via time-marker pretraining.

**Why interesting for dubbing:**
- Timestamp-aware architecture (time-marker insertion during pretraining) → precise word-level timing
- DeepStack cross-layer feature injection → retains prosody and timbre even for accented speech
- Same model also does speaker identification + emotion — could consolidate multiple pipeline stages

**Honest caveats:**
- Published benchmarks are AISHELL-1 (Mandarin) + LibriSpeech (English) — **Tamil/Hindi performance is unknown**
- Released April 2026, Apache 2.0 — no community fine-tuning or production track record yet
- Requires GPU: ~10 GB VRAM for 4B, ~20 GB for 8B
- Does not expose log-probabilities → falls back to PREFERRED_ORDER position (can't win confidence-weighted selection)

**Models:** `OpenMOSS-Team/MOSS-Audio-4B-Instruct` (predictable outputs) · `MOSS-Audio-8B-Instruct` (best quality)  
**GitHub:** https://github.com/OpenMOSS/MOSS-Audio

In [ ]:
# ── MOSS-Audio ASR ────────────────────────────────────────────────────────────
# Swap MOSS_MODEL_ID to MOSS-Audio-8B-Instruct for best quality (needs ~20 GB VRAM).
MOSS_MODEL_ID = 'OpenMOSS-Team/MOSS-Audio-4B-Instruct'

try:
    import torch
    from transformers import AutoModel, AutoProcessor

    device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Loading {MOSS_MODEL_ID} on {device} (first run downloads ~5-10 GB)...')
    t0 = time.time()

    moss_model = AutoModel.from_pretrained(
        MOSS_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=(torch.float16 if device == 'mps' else torch.bfloat16 if device == 'cuda' else torch.float32),
        device_map='auto',
    )
    moss_model.eval()

    moss_proc = AutoProcessor.from_pretrained(
        MOSS_MODEL_ID,
        trust_remote_code=True,
        enable_time_marker=True,
    )
    mel_sr = getattr(getattr(moss_proc, 'config', None), 'mel_sr', 16000)
    print(f'Loaded in {time.time()-t0:.1f}s  |  mel_sr={mel_sr} Hz')

    # Load audio at the model's expected sample rate
    if mel_sr != ASR_SAMPLE_RATE:
        y_moss, _ = librosa.load(VOCALS_WAV, sr=mel_sr, mono=True)
    else:
        y_moss = y_asr

    def slice_moss(start, end):
        s = int(start * mel_sr)
        e = int(end   * mel_sr)
        return y_moss[s:e]

    MOSS_PROMPT = 'Transcribe the speech in this audio. Return only the spoken words, nothing else.'

    segs_moss = []
    t0 = time.time()

    for seg in tqdm(diar_segs, desc='MOSS-Audio ASR'):
        chunk = slice_moss(seg['start'], seg['end'])

        if len(chunk) < mel_sr * 0.1:  # skip < 100 ms
            segs_moss.append({
                'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end'],
                'text': '', 'avg_logprob': None, 'no_speech_prob': None,
            })
            continue

        inputs = moss_proc(text=MOSS_PROMPT, audios=[chunk], return_tensors='pt')
        inputs = {k: v.to(moss_model.device) for k, v in inputs.items()}
        if inputs.get('audio_data') is not None:
            inputs['audio_data'] = inputs['audio_data'].to(moss_model.dtype)
        inputs['audio_input_mask'] = inputs['input_ids'] == moss_proc.audio_token_id

        with torch.no_grad():
            gen_ids = moss_model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                use_cache=True,
            )

        text = moss_proc.decode(
            gen_ids[0, inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip()

        segs_moss.append({
            'speaker':        seg['speaker'],
            'start':          seg['start'],
            'end':            seg['end'],
            'text':           text,
            'avg_logprob':    None,  # MOSS-Audio does not expose token log-probs
            'no_speech_prob': None,
        })

    elapsed = time.time() - t0
    all_results['MOSS-Audio'] = segs_moss
    print(f'MOSS-Audio: {len(segs_moss)} segs in {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    print('Sample outputs:')
    for s in segs_moss[:3]:
        print(f'  [{s["start"]:.1f}-{s["end"]:.1f}s] {s["speaker"]}: {s["text"][:80]}')

except Exception as e:
    _is_oom = 'memory' in str(e).lower() or 'out of mem' in str(e).lower()
    if _is_oom:
        print('MOSS-Audio SKIPPED: not enough VRAM on this machine (needs ~10 GB).')
        print('Run this cell on GCP with a CUDA GPU — everything else continues fine.')
    else:
        print(f'MOSS-Audio ASR FAILED: {e}')
    try:
        import torch; torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
    except Exception: pass
    # MOSS-Audio stays out of all_results — comparison chart will show N/A for it

## Commercial — Gemini ASR (pseudo-GT reference)

Gemini processes the full audio file natively via the Files API. It doesn't use a traditional ASR pipeline — it understands audio holistically, which helps on code-switching, Indic prosody, and overlapping speech. We treat its output as the **pseudo ground truth** for WER ranking of OSS models.


In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or getpass('Gemini API key: ')
genai.configure(api_key=GEMINI_API_KEY)

try:
    lang_name = LANGUAGE_DISPLAY[SOURCE_LANGUAGE]
    print(f'Uploading {VOCALS_WAV} to Gemini Files API...')
    t0 = time.time()
    audio_file = genai.upload_file(VOCALS_WAV, mime_type='audio/wav')
    print(f'Uploaded in {time.time()-t0:.1f}s')

    prompt = f'''Transcribe this {lang_name} audio precisely.
Return a JSON array. Each element:
  "start"  : float, segment start time in seconds
  "end"    : float, segment end time in seconds
  "text"   : string, transcribed text in {lang_name} script (no translation)

Be precise with timestamps. Do not hallucinate text during silence.
Return only valid JSON, no markdown fences.'''

    model_g = genai.GenerativeModel(GEMINI_PRO_MODEL)
    t0 = time.time()
    response = model_g.generate_content([audio_file, prompt])
    elapsed  = time.time() - t0

    raw = response.text.strip()
    if raw.startswith('```'): raw = raw.split('\n', 1)[1].rsplit('```', 1)[0]
    gemini_raw = json.loads(raw)

    segs_gemini = []
    for seg in gemini_raw:
        s = float(seg.get('start', 0))
        e = float(seg.get('end',   s + 2.0))
        sp = best_overlap_speaker(s, e, diar_segs)  # FIX: overlap-based speaker assignment
        segs_gemini.append({'speaker': sp, 'start': s, 'end': e,
                            'text': seg.get('text', '').strip(), 'avg_logprob': None, 'no_speech_prob': None})

    all_results['Gemini-ASR'] = segs_gemini
    print(f'Gemini ASR: {len(segs_gemini)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    genai.delete_file(audio_file.name)
except Exception as e:
    print(f'Gemini ASR FAILED: {e}')


## Commercial — ElevenLabs Scribe v2 & Deepgram Nova-3

Paste API keys when prompted, or press Enter to skip. Both integrate with existing credentials if you tested them in the diarization notebook.

In [ ]:
# ── ElevenLabs Scribe v2 ──────────────────────────────────────────────────────
# Uses existing EL subscription. Multilingual ASR with word-level timestamps
# and native speaker diarization. Strong on Indic languages.
import requests

EL_API_KEY_ASR = os.getenv('ELEVENLABS_API_KEY') or getpass('ElevenLabs API key (Scribe v2, Enter to skip): ')

if not EL_API_KEY_ASR.strip():
    print('[EL Scribe v2] Skipped.')
else:
    try:
        t0 = time.time()
        with open(VOCALS_WAV, 'rb') as f:
            resp = requests.post(
                'https://api.elevenlabs.io/v1/speech-to-text',
                headers={'xi-api-key': EL_API_KEY_ASR},
                files={'audio': (os.path.basename(VOCALS_WAV), f, 'audio/wav')},
                data={
                    'model_id': 'scribe_v1',
                    'language_code': SOURCE_LANGUAGE,
                    'timestamps_granularity': 'word',
                    'diarize': 'true',
                },
            )
        resp.raise_for_status()
        elapsed = time.time() - t0
        data    = resp.json()

        # Group consecutive words by speaker_id into segments
        words = data.get('words', [])
        segs_scribe = []
        if words:
            cur_spk, seg_start, seg_end, seg_words = words[0].get('speaker_id', '0'), words[0]['start'], words[0]['end'], [words[0]['text']]
            for w in words[1:]:
                spk = w.get('speaker_id', '0')
                if spk == cur_spk:
                    seg_end = w['end']
                    seg_words.append(w['text'])
                else:
                    sp = best_overlap_speaker(seg_start, seg_end, diar_segs)
                    segs_scribe.append({'speaker': sp, 'start': seg_start, 'end': seg_end,
                                        'text': ' '.join(seg_words), 'avg_logprob': None, 'no_speech_prob': None})
                    cur_spk, seg_start, seg_end, seg_words = spk, w['start'], w['end'], [w['text']]
            sp = best_overlap_speaker(seg_start, seg_end, diar_segs)
            segs_scribe.append({'speaker': sp, 'start': seg_start, 'end': seg_end,
                                'text': ' '.join(seg_words), 'avg_logprob': None, 'no_speech_prob': None})

        all_results['EL-Scribe-v2'] = segs_scribe
        print(f'EL Scribe v2: {len(segs_scribe)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    except Exception as e:
        print(f'EL Scribe v2 FAILED: {e}')

In [ ]:
# ── Deepgram Nova-3 Multilingual ──────────────────────────────────────────────
# pip install deepgram-sdk
# $200 free credit (no CC). Best word-level timestamps of any commercial API.
# Returns per-utterance confidence scores usable in the model-winner selection.
DEEPGRAM_API_KEY_ASR = os.getenv('DEEPGRAM_API_KEY') or getpass('Deepgram API key (Enter to skip): ')

if not DEEPGRAM_API_KEY_ASR.strip():
    print('[Deepgram Nova-3] Skipped.')
else:
    try:
        from deepgram import DeepgramClient, PrerecordedOptions

        dg_asr = DeepgramClient(DEEPGRAM_API_KEY_ASR)
        with open(VOCALS_WAV, 'rb') as f:
            audio_data = {'buffer': f.read()}

        options = PrerecordedOptions(
            model='nova-3',
            language=SOURCE_LANGUAGE,
            smart_format=True,
            utterances=True,
        )
        t0   = time.time()
        resp = dg_asr.listen.rest.v('1').transcribe_file(audio_data, options)
        elapsed = time.time() - t0

        segs_dg = []
        for utt in (resp.results.utterances or []):
            sp = best_overlap_speaker(utt.start, utt.end, diar_segs)
            segs_dg.append({
                'speaker':        sp,
                'start':          float(utt.start),
                'end':            float(utt.end),
                'text':           utt.transcript,
                'avg_logprob':    utt.confidence,   # 0–1 confidence score used in winner selection
                'no_speech_prob': None,
            })
        all_results['Deepgram-Nova3'] = sorted(segs_dg, key=lambda x: x['start'])
        print(f'Deepgram Nova-3: {len(segs_dg)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    except ImportError:
        print('deepgram-sdk not installed. Run: pip install deepgram-sdk')
    except Exception as e:
        print(f'Deepgram Nova-3 ASR FAILED: {e}')

## Metrics


In [ ]:
from jiwer import wer, cer

# Use Gemini as reference; fall back to Whisper LargeV3
ref_name = 'Gemini-ASR' if 'Gemini-ASR' in all_results else 'Whisper-LargeV3'
ref_text  = ' '.join(s['text'] for s in all_results[ref_name])

rows = []
for name, segs in all_results.items():
    hyp = ' '.join(s['text'] for s in segs)
    # Confidence: fraction of segments with avg_logprob > -1.0 (Whisper threshold)
    lp_vals = [s['avg_logprob'] for s in segs if s.get('avg_logprob') is not None]
    conf_ok = sum(1 for v in lp_vals if v > -1.0) / len(lp_vals) if lp_vals else None
    rows.append({
        'Model':         name,
        'Segments':      len(segs),
        f'Pseudo-WER vs {ref_name}': round(wer(ref_text, hyp), 3) if name != ref_name else 0.0,
        f'Pseudo-CER vs {ref_name}': round(cer(ref_text, hyp), 3) if name != ref_name else 0.0,
        'Conf>-1 %':     round(conf_ok*100, 1) if conf_ok is not None else 'N/A',
        'Total chars':   len(hyp),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


In [ ]:
# ── Model comparison charts (no listening required) ───────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from jiwer import wer, cer

ref_name = 'Gemini-ASR' if 'Gemini-ASR' in all_results else 'Whisper-LargeV3'
if ref_name not in all_results:
    print('Run at least one model and the reference (Gemini or Whisper-LargeV3) first.')
else:
    ref_text = ' '.join(s['text'] for s in all_results[ref_name])

    chart_rows = []
    for name, segs in all_results.items():
        if name == ref_name or not segs:
            continue
        hyp = ' '.join(s['text'] for s in segs)
        lp_vals = [s['avg_logprob'] for s in segs if s.get('avg_logprob') is not None]
        chart_rows.append({
            'model':    name,
            'wer':      round(wer(ref_text, hyp), 3),
            'cer':      round(cer(ref_text, hyp), 3),
            'conf_pct': round(sum(1 for v in lp_vals if v > -1.0) / len(lp_vals) * 100, 1) if lp_vals else None,
        })

    if len(chart_rows) >= 1:
        chart_rows.sort(key=lambda x: x['wer'])
        names = [r['model'] for r in chart_rows]
        wers  = [r['wer']   for r in chart_rows]
        cers  = [r['cer']   for r in chart_rows]

        n_cols = 3 if any(r['conf_pct'] is not None for r in chart_rows) else 2
        fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, max(4, len(names) * 0.55 + 1.5)))

        bars0 = axes[0].barh(names, wers, color='steelblue')
        axes[0].set_xlabel(f'Pseudo-WER vs {ref_name}  (lower = better)')
        axes[0].set_title('Word Error Rate')
        for bar, val in zip(bars0, wers):
            axes[0].text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)

        bars1 = axes[1].barh(names, cers, color='darkorange')
        axes[1].set_xlabel(f'Pseudo-CER vs {ref_name}  (lower = better)')
        axes[1].set_title('Character Error Rate')
        for bar, val in zip(bars1, cers):
            axes[1].text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)

        if n_cols == 3:
            conf_vals = [r['conf_pct'] if r['conf_pct'] is not None else 0 for r in chart_rows]
            bars2 = axes[2].barh(names, conf_vals, color='seagreen')
            axes[2].set_xlabel('% segments with logprob > −1  (higher = more confident)')
            axes[2].set_title('ASR Confidence')
            axes[2].set_xlim(0, 100)

        plt.tight_layout()
        chart_path = os.path.join(TRANSCRIPTION_DIR, 'asr_comparison.png')
        plt.savefig(chart_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Chart saved → {chart_path}')
    else:
        print('Run at least two models to generate a comparison chart.')

In [ ]:
N = 5
for i in range(min(N, len(diar_segs))):
    seg = diar_segs[i]
    print(f'\n[{seg["start"]:.1f}-{seg["end"]:.1f}s] {seg["speaker"]}')
    for name, segs in all_results.items():
        txt = segs[i]['text'] if i < len(segs) else '(missing)'
        lp  = segs[i].get('avg_logprob')
        conf_str = f'  logp={lp:.2f}' if lp is not None else ''
        print(f'  {name:20s}: {txt[:80]}{conf_str}')


In [ ]:
# ── Confidence-weighted selection & save ──────────────────────────────────────
# Per-segment winner: pick the model with the highest avg_logprob.
# If no confidence available, prefer in this order:
# NOTE: MOSS-Audio has no log-probs, so it only wins if all other models are absent.
PREFERRED_ORDER = [
    'Gemini-ASR',
    'EL-Scribe-v2',
    'Deepgram-Nova3',
    'IndicWhisper',
    'Collabora-Whisper-Hindi',
    'Sarvam-1',
    'Canary-1B',
    'MOSS-Audio',        # no log-probs; good benchmark but can't confidence-rank
    'Whisper-LargeV3',
    'WhisperX',
]

final_segs = []
for i, seg in enumerate(diar_segs):
    best_text = best_lp = best_name = None
    for name in PREFERRED_ORDER:
        s = all_results.get(name, [])
        if i >= len(s): continue
        lp = s[i].get('avg_logprob')
        if best_lp is None or (lp is not None and lp > best_lp):
            best_lp   = lp
            best_text = s[i]['text']
            best_name = name
    if best_text is None and all_results:
        first = list(all_results.values())[0]
        best_text = first[i]['text'] if i < len(first) else ''
        best_name = list(all_results.keys())[0]
    final_segs.append({
        'speaker':      seg['speaker'],
        'start':        seg['start'],
        'end':          seg['end'],
        'text':         best_text or '',
        'source_model': best_name,
        'avg_logprob':  best_lp,
    })

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(final_segs, f, indent=2, ensure_ascii=False)

print(f'Saved {len(final_segs)} segments -> {OUT_JSON}')
from collections import Counter
model_counts = Counter(s['source_model'] for s in final_segs)
print('Segments per source model:', dict(model_counts))
for s in final_segs[:5]:
    print(f'  [{s["start"]:.1f}-{s["end"]:.1f}] {s["speaker"]} [{s["source_model"]}]: {s["text"][:60]}')